# SBERT cosine-similarity prior + matrix factorization

This notebook adapts `user_clustering_by_residualvec.ipynb` for a **classical-only** test of whether an SBERT semantic prior improves matrix factorization (MF).

The residual-vector clustering, residual bins, and conditional table $P(E\mid T,g)$ are replaced by:

$$
\text{synopsis}\xrightarrow{\text{SBERT}}e_i,
\qquad S_{ij}=\cos(e_i,e_j),
\qquad P_\beta(j\mid i)=\operatorname{softmax}_{j\ne i}(\beta S_{ij}).
$$



## Imports, reproducibility, and configuration

In [1]:
from __future__ import annotations

import copy
import gc
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from torch.utils.data import DataLoader, Dataset

SEED = 42
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_all_seeds(SEED)
print("Device:", DEVICE)

Device: mps


In [ ]:
# Expected files:
# The synopsis CSV must contain item_id plus synopsis.

ROOT = Path.cwd()
if not (ROOT / "Data/ml-100k/u.data").exists():
    for candidate in (ROOT.parent, ROOT.parent.parent):
        if (candidate / "Data/ml-100k/u.data").exists():
            ROOT = candidate
            break

# DATA PATHS
RATINGS_PATH = ROOT / "Data/ml-100k/u.data"
MOVIES_PATH = ROOT / "Data/ml-100k/movies_with_synopsis_complete.csv"

# Ouput directory for SBERT embeddings and results
OUTPUT_DIR = ROOT / "sbert"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# SBERT MODEL
SBERT_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_CACHE = OUTPUT_DIR / "sbert_embeddings.npz"

# Grid search results path
GRID_RESULTS_PATH = OUTPUT_DIR / "cos_sim_validation_grid.csv"
RESULTS_PATH = OUTPUT_DIR / "cos_sim_results.csv"

MF_DIM = 64
BATCH_SIZE = 1024
EPOCHS = 30
PATIENCE = 5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
MIN_INTERACTIONS = 5

# Tune using validation only. QUICK_RUN provides a fast pipeline check.
BETA_GRID = [1.0, 5.0, 10.0, 20.0]
LAMBDA_GRID = [0.0, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1]
FINAL_SEEDS = [0, 1, 2, 3, 4]
QUICK_RUN = False

if QUICK_RUN:
    BETA_GRID = [5.0]
    LAMBDA_GRID = [0.0, 1e-2]
    FINAL_SEEDS = [0]
    EPOCHS = 2
    PATIENCE = 2

print("Ratings:", RATINGS_PATH)
print("Synopses:", MOVIES_PATH)
print("Outputs:", OUTPUT_DIR)

Ratings: /Users/kiradavidoff/code/quantum_correlations/Data/ml-100k/u.data
Synopses: /Users/kiradavidoff/code/quantum_correlations/Data/ml-100k/movies_with_synopsis_complete.csv
Outputs: /Users/kiradavidoff/code/quantum_correlations/sbert


## Load MovieLens and align synopsis item IDs

Here I had created a csv containing alll the synopsis of the movies (movies_with_synopsis_complete.csv). 

In [3]:
ratings = pd.read_csv(
    RATINGS_PATH,
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"],
)

movies = pd.read_csv(MOVIES_PATH)
if "item_id" not in movies.columns and "movieId" in movies.columns:
    movies = movies.rename(columns={"movieId": "item_id"})

required = {"item_id", "synopsis"}
missing = required - set(movies.columns)
if missing:
    raise ValueError(f"Synopsis file is missing columns: {sorted(missing)}")

movies = movies.copy()
movies["item_id"] = pd.to_numeric(movies["item_id"], errors="raise").astype(int)
movies["synopsis"] = movies["synopsis"].fillna("").astype(str).str.strip()

if movies["item_id"].duplicated().any():
    duplicates = movies.loc[movies["item_id"].duplicated(), "item_id"].tolist()[:10]
    raise ValueError(f"Duplicate item IDs in synopsis file, e.g. {duplicates}")

rating_item_ids = set(ratings["item_id"].astype(int))
synopsis_item_ids = set(movies["item_id"])
missing_synopses = sorted(rating_item_ids - synopsis_item_ids)
blank_synopses = movies.loc[movies["synopsis"].eq(""), "item_id"].tolist()

print(f"Ratings: {len(ratings):,}; users: {ratings.user_id.nunique():,}; items: {ratings.item_id.nunique():,}")
print(f"Synopsis rows: {len(movies):,}")
print(f"Rating items missing from synopsis file: {len(missing_synopses)}")
print(f"Blank synopses: {len(blank_synopses)}")
if missing_synopses:
    print("First missing IDs:", missing_synopses[:10])

Ratings: 100,000; users: 943; items: 1,682
Synopsis rows: 1,682
Rating items missing from synopsis file: 0
Blank synopses: 0


## Warm-start temporal split

Function umported from Freya's work to make the split of data temporal

In [ ]:
def split_by_user_time(df, train_ratio=0.8, val_ratio=0.1, min_n=5):
    train_parts, val_parts, test_parts = [], [], []
    dropped_users = []

    for uid, group in df.sort_values(["user_id", "timestamp"]).groupby("user_id", sort=False):
        group = group.sort_values("timestamp", kind="stable")
        n = len(group)
        if n < min_n:
            dropped_users.append(uid)
            continue

        n_train = max(1, int(np.floor(n * train_ratio)))
        n_val = max(1, int(np.floor(n * val_ratio)))
        if n_train + n_val >= n:
            n_train, n_val = n - 2, 1

        train_parts.append(group.iloc[:n_train])
        val_parts.append(group.iloc[n_train:n_train + n_val])
        test_parts.append(group.iloc[n_train + n_val:])

    return (
        pd.concat(train_parts, ignore_index=True),
        pd.concat(val_parts, ignore_index=True),
        pd.concat(test_parts, ignore_index=True),
        dropped_users,
    )

train, val, test, dropped_users = split_by_user_time(
    ratings, min_n=MIN_INTERACTIONS
)

train_users = set(train.user_id)
train_items = set(train.item_id)
synopsis_items = set(movies.item_id)

def warm_and_text_filter(frame):
    """
    Keep ratings for users and movies present in training,
    where the movie also has a synopsis for SBERT.
    Returns the filtered DataFrame and number of removed rows.
    """

    mask = (
        frame.user_id.isin(train_users)
        & frame.item_id.isin(train_items)
        & frame.item_id.isin(synopsis_items)
    )
    return frame.loc[mask].reset_index(drop=True), int((~mask).sum())

train_eval, train_removed = warm_and_text_filter(train)
val_eval, val_removed = warm_and_text_filter(val)
test_eval, test_removed = warm_and_text_filter(test)

print({
    "train": len(train_eval), "validation": len(val_eval), "test": len(test_eval),
    "train_removed": train_removed, "validation_removed": val_removed,
    "test_removed": test_removed, "dropped_users": len(dropped_users),
})
assert len(train_eval) and len(val_eval) and len(test_eval)

{'train': 79619, 'validation': 9559, 'test': 10736, 'train_removed': 0, 'validation_removed': 37, 'test_removed': 49, 'dropped_users': 0}


## SBERT synopsis embeddings

In [16]:
from sentence_transformers import SentenceTransformer

def load_or_encode_sbert(movie_frame, cache_path, model_name):
    expected_ids = movie_frame["item_id"].to_numpy(dtype=np.int64)

    # Load cached embeddings if available and aligned with the expected item IDs.
    if cache_path.exists():
        cached = np.load(cache_path)
        cached_ids = cached["item_ids"].astype(np.int64)
        embeddings = cached["embeddings"].astype(np.float32)
        if np.array_equal(cached_ids, expected_ids):
            print("Loaded aligned embedding cache:", cache_path)
            return embeddings
        print("Ignoring cache because its item-ID order does not match the synopsis CSV.")

    # Encode synopses using SBERT and cache the embeddings.
    encoder = SentenceTransformer(model_name, device=str(DEVICE))
    embeddings = encoder.encode(
        movie_frame["synopsis"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)
    np.savez_compressed(cache_path, item_ids=expected_ids, embeddings=embeddings)
    return embeddings

all_embeddings = load_or_encode_sbert(movies, EMBEDDING_CACHE, SBERT_MODEL_NAME)

# Validate that the embeddings are aligned with the movies DataFrame and contain finite values.
assert all_embeddings.shape[0] == len(movies)
assert np.all(np.isfinite(all_embeddings))


Loaded aligned embedding cache: /Users/kiradavidoff/code/quantum_correlations/sbert/sbert_embeddings.npz


## Cosine similarities

$$

S_{ij}= EE^T \\
\\
\\
\text{E : Embedings Matrix (normalised)}
$$

In [ ]:
# Create mappings from item and user IDs to indices
movie_row = {int(item_id): row for row, item_id in enumerate(movies.item_id)}
item_ids = sorted(set(train_eval.item_id))
user_ids = sorted(set(train_eval.user_id))
ITEM_TO_IDX = {item_id: idx for idx, item_id in enumerate(item_ids)}
USER_TO_IDX = {user_id: idx for idx, user_id in enumerate(user_ids)}

train_rows = np.array([movie_row[item_id] for item_id in item_ids], dtype=np.int64)
item_embeddings = all_embeddings[train_rows]
item_embeddings /= np.maximum(np.linalg.norm(item_embeddings, axis=1, keepdims=True), 1e-12)


# Make sure everything is normaliseed (not strictly necessary since we normalized above, but I want to check it)
norms = np.linalg.norm(item_embeddings, axis=1)

if not np.allclose(norms, 1.0):
    # Normalize embeddings if necessary
    item_embeddings = item_embeddings / np.maximum(
        norms[:, None], 1e-12
    )

# Compute the cosine similarity matrix for the item embeddings
cosine_similarity = np.clip(item_embeddings @ item_embeddings.T, -1.0, 1.0).astype(np.float32)

assert cosine_similarity.shape == (len(item_ids), len(item_ids))
assert all(item_ids[ITEM_TO_IDX[item_id]] == item_id for item_id in item_ids)
print("Train-aligned SBERT matrix:", item_embeddings.shape)
print("Train-aligned cosine matrix:", cosine_similarity.shape)

In [25]:
norms = np.linalg.norm(item_embeddings, axis=1)

if not np.allclose(norms, 1.0):
    # Normalize embeddings if necessary
    item_embeddings = item_embeddings / np.maximum(
        norms[:, None], 1e-12
    )

# Calculate pairwise cosine similarity
cosine_similarity = np.clip(
    item_embeddings @ item_embeddings.T,
    -1.0,
    1.0
).astype(np.float32)

## Semantic prior and null controls

PRIOR:

$$
\boxed{
P(j\mid i)=
\frac{\exp(\beta S_{ij})}
{\sum_{k\ne i}\exp(\beta S_{ik})}
\quad\text{for }j\ne i
}
$$

The parameter $\beta$ controls how strongly  prior favours semantically similar movies



In [39]:
def semantic_prior_from_cosine(similarity, beta):
    # Row-stochastic P(j|i), excluding self, with stable softmax.
    logits = beta * similarity.astype(np.float64).copy()
    np.fill_diagonal(logits, -np.inf)
    row_max = np.max(logits, axis=1, keepdims=True)
    weights = np.exp(logits - row_max)
    np.fill_diagonal(weights, 0.0)
    prior = weights / weights.sum(axis=1, keepdims=True)
    return prior.astype(np.float32)


example_prior = semantic_prior_from_cosine(cosine_similarity, beta=BETA_GRID[0])


## Matrix factorization model and datasets

In [7]:
class MFDataset(Dataset):
    def __init__(self, frame):
        self.users = torch.tensor(
            [USER_TO_IDX[int(x)] for x in frame.user_id], dtype=torch.long
        )
        self.items = torch.tensor(
            [ITEM_TO_IDX[int(x)] for x in frame.item_id], dtype=torch.long
        )
        self.ratings = torch.tensor(frame.rating.to_numpy(), dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, index):
        return self.users[index], self.items[index], self.ratings[index]


class MF(nn.Module):
    def __init__(self, n_users, n_items, k=64, mu=3.5):
        super().__init__()
        self.P = nn.Embedding(n_users, k)
        self.Q = nn.Embedding(n_items, k)
        self.bu = nn.Embedding(n_users, 1)
        self.bi = nn.Embedding(n_items, 1)
        self.mu = nn.Parameter(torch.tensor(float(mu)))

        nn.init.normal_(self.P.weight, std=0.05)
        nn.init.normal_(self.Q.weight, std=0.05)
        nn.init.zeros_(self.bu.weight)
        nn.init.zeros_(self.bi.weight)

    def forward(self, users, items):
        dot = (self.P(users) * self.Q(items)).sum(dim=1)
        return self.mu + self.bu(users).squeeze(1) + self.bi(items).squeeze(1) + dot


train_ds, val_ds, test_ds = map(MFDataset, (train_eval, val_eval, test_eval))

def make_loaders(seed):
    generator = torch.Generator().manual_seed(seed)
    return (
        DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                   generator=generator, num_workers=0),
        DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0),
        DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0),
    )

## Semantic graph loss

For the item-factor matrix $Q$, the prior penalty is the expected squared distance

$$
\mathcal L_{\mathrm{prior}}=
\frac{1}{N}\sum_i\sum_j P_\beta(j\mid i)\lVert q_i-q_j\rVert_2^2.
$$



In [ ]:
def semantic_graph_loss(model, prior_tensor):
    q = model.Q.weight
    squared_norm = q.square().sum(dim=1)

    row_term = (prior_tensor.sum(dim=1) * squared_norm).mean()
    col_term = (prior_tensor.sum(dim=0) * squared_norm).mean()

    interaction = (q * (prior_tensor @ q)).sum() / q.shape[0]
    return row_term + col_term - 2.0 * interaction  #||qi-qj||^2= ||qi||^2+||qj||^2-2*qi.qj


@torch.no_grad()
def evaluate_model(model, loader, clip=True):
    model.eval()

    predictions, targets = [], []

    for users, items, ratings_batch in loader:
        users, items = users.to(DEVICE), items.to(DEVICE)
        pred = model(users, items)
        if clip:
            pred = pred.clamp(1.0, 5.0)
        predictions.append(pred.cpu().numpy())
        targets.append(ratings_batch.numpy())


    predictions = np.concatenate(predictions)
    targets = np.concatenate(targets)

    return {
        "rmse": float(np.sqrt(np.mean((predictions - targets) ** 2))),
        "mae": float(np.mean(np.abs(predictions - targets))),
    }

## Training with best-validation checkpoint restoration

In [33]:
def train_model(prior, lambda_prior, seed, epochs=EPOCHS, patience=PATIENCE, verbose=1):
    # 1. Initialization and Setup
    set_all_seeds(seed)
    train_loader, val_loader, test_loader = make_loaders(seed)

    model = MF(
        n_users=len(USER_TO_IDX),
        n_items=len(ITEM_TO_IDX),
        k=MF_DIM,
        mu=float(train_eval.rating.mean()),
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    mse_fn = nn.MSELoss()

    # Prepare the prior tensor once before training starts
    prior_tensor = (
        torch.as_tensor(prior, dtype=torch.float32, device=DEVICE)
        if prior is not None and lambda_prior > 0 else None
    )

    history = []
    best_state, best_val_rmse, stale = None, float("inf"), 0

    # 2. Training Loop across Epochs
    for epoch in range(1, epochs + 1):
        model.train()
        rating_loss_sum = 0.0
        prior_loss_sum = 0.0
        n_examples = 0

        # Batch-level training
        for users, items, targets in train_loader:
            users, items, targets = users.to(DEVICE), items.to(DEVICE), targets.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            # Compute standard rating loss (MSE)
            mse = mse_fn(model(users, items), targets)

            # Compute prior regularization loss (if prior exists)
            if prior_tensor is not None:
                graph = semantic_graph_loss(model, prior_tensor)
                total_loss = mse + (lambda_prior * graph)
                prior_loss_sum += float(graph.detach().cpu()) * len(targets)
            else:
                total_loss = mse

            # Single backward pass and joint optimizer step
            total_loss.backward()
            optimizer.step()

            # Track metrics
            rating_loss_sum += mse.item() * len(targets)
            n_examples += len(targets)

        # Calculate epoch-level averages
        train_rmse = float(np.sqrt(rating_loss_sum / n_examples))
        avg_prior = prior_loss_sum / n_examples if prior_tensor is not None else 0.0

        #  Validation and Metrics Evaluation
        val_metrics = evaluate_model(model, val_loader)
        row = {
            "epoch": epoch,
            "train_rmse": train_rmse,
            "prior_loss": avg_prior,
            "val_rmse": val_metrics["rmse"],
            "val_mae": val_metrics["mae"],
        }
        history.append(row)

        if verbose:
            print(row)

        # Early Stopping Check
        if val_metrics["rmse"] < best_val_rmse - 1e-7:
            best_val_rmse = val_metrics["rmse"]
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                print(f"Early stopping triggered at epoch {epoch}")
                break

    # Restore Best Model and Return
    assert best_state is not None, "Training failed to find a valid model state."
    model.load_state_dict(best_state)
    restored_val = evaluate_model(model, val_loader)

    return model, pd.DataFrame(history), restored_val, test_loader

## Validation-only grid search


In [35]:
grid_rows = []
grid_histories = {}

# One baseline, independent of beta.
start = time.time()
baseline_model, baseline_history, baseline_val, _ = train_model(
    prior=None, lambda_prior=0.0, seed=SEED
)
grid_rows.append({
    "condition": "baseline", "beta": np.nan, "lambda_prior": 0.0,
    "val_rmse": baseline_val["rmse"], "val_mae": baseline_val["mae"],
    "best_epoch": int(baseline_history.loc[baseline_history.val_rmse.idxmin(), "epoch"]),
    "seconds": time.time() - start,
})
grid_histories[("baseline", None, 0.0)] = baseline_history
del baseline_model

for beta in BETA_GRID:
    real_prior = semantic_prior_from_cosine(cosine_similarity, beta)
    for lambda_prior in [x for x in LAMBDA_GRID if x > 0]:
        print(f"Validation grid: beta={beta}, lambda={lambda_prior}")
        start = time.time()
        model, history, val_metrics, _ = train_model(
            prior=real_prior, lambda_prior=lambda_prior, seed=SEED
        )
        grid_rows.append({
            "condition": "sbert", "beta": beta, "lambda_prior": lambda_prior,
            "val_rmse": val_metrics["rmse"], "val_mae": val_metrics["mae"],
            "best_epoch": int(history.loc[history.val_rmse.idxmin(), "epoch"]),
            "seconds": time.time() - start,
        })
        grid_histories[("sbert", beta, lambda_prior)] = history
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

grid_results = pd.DataFrame(grid_rows).sort_values("val_rmse").reset_index(drop=True)
grid_results.to_csv(GRID_RESULTS_PATH, index=False)
display(grid_results)

{'epoch': 1, 'train_rmse': 1.0948311106955264, 'prior_loss': 0.0, 'val_rmse': 1.1618996858596802, 'val_mae': 0.9628731608390808}
{'epoch': 2, 'train_rmse': 1.0673723896511955, 'prior_loss': 0.0, 'val_rmse': 1.1409952640533447, 'val_mae': 0.9437044858932495}
{'epoch': 3, 'train_rmse': 1.0399914640052037, 'prior_loss': 0.0, 'val_rmse': 1.1190199851989746, 'val_mae': 0.9227701425552368}
{'epoch': 4, 'train_rmse': 1.007882009375707, 'prior_loss': 0.0, 'val_rmse': 1.093899130821228, 'val_mae': 0.8963732123374939}
{'epoch': 5, 'train_rmse': 0.9675419906294337, 'prior_loss': 0.0, 'val_rmse': 1.0636783838272095, 'val_mae': 0.8633706569671631}
{'epoch': 6, 'train_rmse': 0.9215721904882526, 'prior_loss': 0.0, 'val_rmse': 1.0330520868301392, 'val_mae': 0.8301979303359985}
{'epoch': 7, 'train_rmse': 0.8765863483155509, 'prior_loss': 0.0, 'val_rmse': 1.0085394382476807, 'val_mae': 0.8053678870201111}
{'epoch': 8, 'train_rmse': 0.8354589817948643, 'prior_loss': 0.0, 'val_rmse': 0.9902763962745667, '

,condition,beta,lambda_prior,val_rmse,val_mae,best_epoch,seconds
0,sbert,20.0,0.100,0.940840,0.741244,19,35.675297
1,sbert,10.0,0.100,0.940883,0.741673,19,36.288646
2,sbert,5.0,0.100,0.941475,0.742285,19,36.649278
3,sbert,1.0,0.100,0.941753,0.742496,19,35.736874
4,sbert,20.0,0.030,0.951631,0.752248,15,29.651043
5,sbert,10.0,0.030,0.951871,0.752611,15,31.077685
6,sbert,5.0,0.030,0.952199,0.752931,15,30.101687
7,sbert,1.0,0.030,0.952339,0.753049,15,31.598439
8,sbert,20.0,0.010,0.958116,0.758005,14,28.174232
9,sbert,10.0,0.010,0.958291,0.758180,14,29.038647


## Select the setting 

# ADD BASELINE

In [36]:
best_sbert = (
    grid_results.query("condition == 'sbert'")
    .sort_values("val_rmse")
    .iloc[0]
)
BEST_BETA = float(best_sbert.beta)
BEST_LAMBDA = float(best_sbert.lambda_prior)

print("Validation-selected SBERT setting:", {
    "beta": BEST_BETA,
    "lambda_prior": BEST_LAMBDA,
    "val_rmse": float(best_sbert.val_rmse),
    "best_epoch": int(best_sbert.best_epoch),
})

Validation-selected SBERT setting: {'beta': 20.0, 'lambda_prior': 0.1, 'val_rmse': 0.9408395886421204, 'best_epoch': 19}


## Final multi-seed baseline, SBERT, and null-control evaluation


In [37]:
EPOCHS = 20

# No need to redefine set_all_seeds, it's already defined above
# Just verify our final selected hyperparameters

print("Selected setting:")
print({
    "beta": BEST_BETA,
    "lambda_prior": BEST_LAMBDA,
    "epochs": EPOCHS,
    "seeds": FINAL_SEEDS,
})

Selected setting:
{'beta': 20.0, 'lambda_prior': 0.1, 'epochs': 20, 'seeds': [0, 1, 2, 3, 4]}


In [ ]:
def make_controls(real_prior, seed=42):
    """Create null-control priors: shuffled, uniform, and random destinations."""
    rng = np.random.default_rng(seed)
    n = real_prior.shape[0]

    # Shuffled: relabel items while preserving graph structure
    permutation = rng.permutation(n)
    shuffled = real_prior[np.ix_(permutation, permutation)].copy()

    # Uniform: flat distribution without semantic bias
    uniform = np.ones((n, n), dtype=np.float32)
    np.fill_diagonal(uniform, 0.0)
    uniform /= uniform.sum(axis=1, keepdims=True)

    # Random: preserve weight multiset per item, randomize destinations
    random_prior = np.zeros_like(real_prior)
    for i in range(n):
        destinations = np.delete(np.arange(n), i)
        weights = np.delete(real_prior[i], i).copy()
        rng.shuffle(weights)
        random_prior[i, destinations] = weights

    # Validate that all priors are row-stochastic, non-negative, and have zero diagonal
    for name, prior in {
        "real": real_prior, "shuffled": shuffled,
        "uniform": uniform, "random": random_prior,
    }.items():
        assert np.allclose(prior.sum(axis=1), 1.0, atol=1e-5), name
        assert np.allclose(np.diag(prior), 0.0), name
        assert np.all(prior >= 0), name

    return {"real": real_prior, "shuffled": shuffled,
            "uniform": uniform, "random": random_prior}

In [38]:
real_prior = semantic_prior_from_cosine(cosine_similarity, BEST_BETA)
priors = make_controls(real_prior, seed=42)

conditions = {
    "Baseline MF": (None, 0.0),
    "MF + SBERT prior": (priors["real"], BEST_LAMBDA),
    "MF + shuffled prior": (priors["shuffled"], BEST_LAMBDA),
    "MF + uniform prior": (priors["uniform"], BEST_LAMBDA),
    "MF + random prior": (priors["random"], BEST_LAMBDA),
}

final_rows = []
for seed in FINAL_SEEDS:
    for condition, (prior, strength) in conditions.items():
        print(f"seed={seed} | {condition}")
        model, history, val_metrics, test_loader = train_model(
            prior=prior, lambda_prior=strength, seed=seed
        )
        # Evaluate on test set after all choices are fixed
        test_metrics = evaluate_model(model, test_loader)
        final_rows.append({
            "condition": condition,
            "seed": seed,
            "beta": BEST_BETA if prior is not None else np.nan,
            "lambda_prior": strength,
            "best_epoch": int(history.loc[history.val_rmse.idxmin(), "epoch"]),
            "val_rmse": val_metrics["rmse"],
            "val_mae": val_metrics["mae"],
            "test_rmse": test_metrics["rmse"],
            "test_mae": test_metrics["mae"],
        })
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

final_results = pd.DataFrame(final_rows)
final_results.to_csv(RESULTS_PATH, index=False)
display(final_results)

seed=0 | Baseline MF
{'epoch': 1, 'train_rmse': 1.0948694692618033, 'prior_loss': 0.0, 'val_rmse': 1.161524772644043, 'val_mae': 0.9630987048149109}
{'epoch': 2, 'train_rmse': 1.0674891413006455, 'prior_loss': 0.0, 'val_rmse': 1.1412732601165771, 'val_mae': 0.9445079565048218}
{'epoch': 3, 'train_rmse': 1.0401260983502494, 'prior_loss': 0.0, 'val_rmse': 1.120887041091919, 'val_mae': 0.9243446588516235}
{'epoch': 4, 'train_rmse': 1.0078800917266455, 'prior_loss': 0.0, 'val_rmse': 1.0965228080749512, 'val_mae': 0.8989251255989075}
{'epoch': 5, 'train_rmse': 0.9673947982934413, 'prior_loss': 0.0, 'val_rmse': 1.0675718784332275, 'val_mae': 0.867083728313446}
{'epoch': 6, 'train_rmse': 0.9214586483630859, 'prior_loss': 0.0, 'val_rmse': 1.038092017173767, 'val_mae': 0.8356600403785706}
{'epoch': 7, 'train_rmse': 0.8765432678582042, 'prior_loss': 0.0, 'val_rmse': 1.0145258903503418, 'val_mae': 0.8109078407287598}
{'epoch': 8, 'train_rmse': 0.834852509528112, 'prior_loss': 0.0, 'val_rmse': 0.9

,condition,seed,beta,lambda_prior,best_epoch,val_rmse,val_mae,test_rmse,test_mae
0,Baseline MF,0,NaN,0.0,13,0.973855,0.771386,1.012987,0.807007
1,MF + SBERT prior,0,20.0,0.1,17,0.949822,0.749995,0.987579,0.785462
2,MF + shuffled prior,0,20.0,0.1,17,0.952720,0.752627,0.991558,0.788514
3,MF + uniform prior,0,20.0,0.1,17,0.950752,0.751054,0.989958,0.787823
4,MF + random prior,0,20.0,0.1,17,0.951420,0.751687,0.990813,0.788348
5,Baseline MF,1,NaN,0.0,13,0.968027,0.764573,1.018202,0.813099
6,MF + SBERT prior,1,20.0,0.1,18,0.943947,0.744609,0.990466,0.788203
7,MF + shuffled prior,1,20.0,0.1,18,0.946549,0.746297,0.994931,0.791803
8,MF + uniform prior,1,20.0,0.1,18,0.945334,0.745899,0.993064,0.790723
9,MF + random prior,1,20.0,0.1,18,0.945602,0.746420,0.994332,0.791766


## Aggregate metrics and paired seed-level comparisons

In [40]:
condition_order = list(conditions)
summary = (
    final_results.groupby("condition", sort=False)
    .agg(
        mean_val_rmse=("val_rmse", "mean"),
        std_val_rmse=("val_rmse", "std"),
        mean_test_rmse=("test_rmse", "mean"),
        std_test_rmse=("test_rmse", "std"),
        mean_test_mae=("test_mae", "mean"),
        std_test_mae=("test_mae", "std"),
        mean_best_epoch=("best_epoch", "mean"),
    )
    .reindex(condition_order)
    .reset_index()
)

baseline_rmse = summary.loc[
    summary.condition.eq("Baseline MF"), "mean_test_rmse"
].iloc[0]
summary["test_rmse_gain_vs_baseline"] = baseline_rmse - summary["mean_test_rmse"]
display(summary.round(6))


,condition,mean_val_rmse,std_val_rmse,mean_test_rmse,std_test_rmse,mean_test_mae,std_test_mae,mean_best_epoch,test_rmse_gain_vs_baseline
0,Baseline MF,0.972485,0.004136,1.016384,0.002613,0.810908,0.002575,12.8,0.000000
1,MF + SBERT prior,0.947932,0.003202,0.989385,0.001843,0.786880,0.001365,17.6,0.026999
2,MF + shuffled prior,0.950749,0.003029,0.993102,0.001681,0.790057,0.001385,17.2,0.023283
3,MF + uniform prior,0.948877,0.002915,0.991308,0.001620,0.788916,0.001260,17.6,0.025076
4,MF + random prior,0.949214,0.002969,0.992180,0.001697,0.789519,0.001441,17.6,0.024204


In [41]:
paired = final_results.pivot(index="seed", columns="condition", values="test_rmse")
for condition in condition_order[1:]:
    paired[f"gain: {condition}"] = paired["Baseline MF"] - paired[condition]
paired["SBERT gain over shuffled"] = (
    paired["MF + shuffled prior"] - paired["MF + SBERT prior"]
)
paired["SBERT gain over random"] = (
    paired["MF + random prior"] - paired["MF + SBERT prior"]
)
display(paired.round(6))

condition,Baseline MF,MF + SBERT prior,MF + random prior,MF + shuffled prior,MF + uniform prior,gain: MF + SBERT prior,gain: MF + shuffled prior,gain: MF + uniform prior,gain: MF + random prior,SBERT gain over shuffled,SBERT gain over random
seed,,,,,,,,,,,
0,1.012987,0.987579,0.990813,0.991558,0.989958,0.025407,0.021429,0.023029,0.022174,0.003978,0.003233
1,1.018202,0.990466,0.994332,0.994931,0.993064,0.027735,0.023270,0.025138,0.023869,0.004465,0.003866
2,1.014671,0.987810,0.990388,0.991664,0.989618,0.026861,0.023007,0.025054,0.024284,0.003854,0.002577
3,1.016596,0.989116,0.991877,0.992486,0.990975,0.027480,0.024110,0.025621,0.024719,0.003370,0.002761
4,1.019466,0.991955,0.993491,0.994870,0.992927,0.027510,0.024596,0.026539,0.025975,0.002914,0.001535
